# AI Resume Analyzer & ATS Score Predictor

### End-to-End NLP + LLM Project

Technologies:
- Python
- NLP
- Sentence Transformers
- Gemini LLM
- Streamlit

# Phase 1: Importing Libraries

In [ ]:
import pdfplumber
import pandas as pd
import numpy as np
import spacy

## Phase 2 : Resume Parsing

In [ ]:
def extract_text_from_pdf(pdf_path):

    try:

        full_text = ""

        with pdfplumber.open(pdf_path) as pdf:

            for page in pdf.pages:

                text = page.extract_text()

                if text:

                    full_text += text + "\n"

        return full_text

    except Exception as e:

        print(f"Error: {e}")

        return None

In [ ]:
# Testing the function :
resume_text = extract_text_from_pdf("sample_resume.pdf")
print(resume_text)

In [ ]:
# Test 1 (Correct File)
resume_text = extract_text_from_pdf("sample_resume.pdf")

print(resume_text)

In [ ]:
# Test 1 (Wrong File)
wrong_resume = extract_text_from_pdf("abc.pdf")

In [ ]:
clean_text = "I am studying machinelearning and deeplearning."
# Normalize common merged technical terms

clean_text = clean_text.replace("machinelearning", "machine learning")
clean_text = clean_text.replace("deeplearning", "deep learning")
clean_text = clean_text.replace("naturallanguageprocessing", "natural language processing")
clean_text = clean_text.replace("scikitlearn", "scikit-learn")
clean_text = clean_text.replace("computervision", "computer vision")
clean_text = clean_text.replace("promptengineering", "prompt engineering")
clean_text = clean_text.replace("powerbi", "power bi")
clean_text = clean_text.replace("githubcom", "github")

print(clean_text)

# Phase 3: Text Preprocessing

Convert Text to Lowercase

In [ ]:
print("Python" == "python")

In [ ]:
print(resume_text)

In [ ]:
resume_text = extract_text_from_pdf("sample_resume.pdf")

print(resume_text)

In [ ]:
resume_text = extract_text_from_pdf("sample_resume.pdf")

In [ ]:
clean_text = resume_text.lower()

print(clean_text)

Removing Extra Spaces

In [ ]:
# See the Hidden Characters
print(repr(clean_text[:500]))

In [ ]:
# Clean Extra Spaces
clean_text = " ".join(clean_text.split())
print(clean_text[:1000])

Remove Special Characters

In [ ]:
import re

In [ ]:
sample = "Python, SQL!!! TensorFlow (Keras)"
clean_sample = re.sub(r"[^\w\s]", "", sample)
print(clean_sample)

In [ ]:
# No special characters :
clean_text = re.sub(r"[^\w\s]", "", clean_text)
print(clean_text[:1000])

Tokenization

In [ ]:
tokens = clean_text.split()
print(tokens[:100])

In [ ]:
print("Total Tokens:", len(tokens))

# Phase 4 : Skill Extraction

In [ ]:
skills = [

    # Programming
    "python",
    "sql",
    "java",
    "c++",

    # Machine Learning
    "machine learning",
    "machinelearning",
    "deep learning",
    "deeplearning",

    # NLP
    "nlp",
    "natural language processing",
    "naturallanguageprocessing",
    "transformers",
    "bert",
    "llm",
    "rag",
    "langchain",

    # Computer Vision
    "computer vision",
    "computervision",
    "opencv",
    "cnn",

    # Libraries
    "tensorflow",
    "keras",
    "scikit-learn",
    "scikitlearn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "xgboost",
    "adaboost",

    # BI
    "power bi",
    "powerbi",
    "tableau",

    # Deployment
    "streamlit",
    "fastapi",

    # Cloud
    "aws",
    "azure",
    "ibm watsonx",
    "watsonx",

    # Databases
    "mysql",
    "sqlite",

    # Version Control
    "git",
    "github"
]

In [ ]:
# Extract Skills
found_skills = []

for skill in skills:
    if skill in clean_text:
        found_skills.append(skill)

found_skills = sorted(set(found_skills))

print(found_skills)

Skill Extraction using spaCy PhraseMatcher

In [ ]:
# Import spaCy
import spacy
from spacy.matcher import PhraseMatcher

In [ ]:
# Load the NLP Model
nlp = spacy.load("en_core_web_sm")
print("spaCy Loaded Successfully!")

In [ ]:
# Creating PhraseMatcher
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

In [ ]:
# Convert Skills into spaCy Patterns
patterns = [nlp.make_doc(skill) for skill in skills]

In [ ]:
print(type(patterns[0]))

In [ ]:
# Add Skill Patterns to PhraseMatcher
matcher.add("SKILLS", patterns)
print("Skill patterns added successfully!")

In [ ]:
# Finding Skill Matches in Resume

doc = nlp(clean_text)
matches = matcher(doc)
matched_skills = []
for match_id, start, end in matches:
    matched_skills.append(doc[start:end].text)
matched_skills = sorted(set(matched_skills))
print("Matched Skills:")
print(matched_skills)

## Observation

The PhraseMatcher implementation was successfully verified using sample text.

However, the extracted resume PDF merges multiple words together (e.g., "usingpythonscikitlearntensorflow..."), causing spaCy to treat them as a single token.

As a result, token-based phrase matching cannot detect individual skills from this PDF. This is a limitation of the PDF text extraction process rather than the PhraseMatcher itself.

To overcome this limitation, the next implementation uses Sentence Transformers for semantic skill extraction.

In [ ]:
# Semantic Skill Extraction using Sentence Transformers

# Instead of checking whether a skill exactly matches the text (like "python"), we want the model to understand meaning.
# What are Sentence Transformers?
# A Sentence Transformer is a pre-trained Deep Learning model that converts text into embeddings (vectors of numbers).

In [ ]:
# Semantic Skill Extraction using Sentence Transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
# Load the model 

model = SentenceTransformer("all-MiniLM-L6-v2")                  
print("Sentence Transformer Loaded Successfully!")

It is a pre-trained Sentence Transformer model from Hugging Face.

It is:
Small
Fast
Very accurate
Commonly used in semantic search applications

The first time you run it, it may take a minute or two because it downloads the model.

In [ ]:
# Create Skill Embeddings

skill_embeddings = model.encode(skills)

print("Total Skill Embeddings:", len(skill_embeddings))

In [ ]:
# Convert Resume into an Embedding

resume_embedding = model.encode([resume_text])
print(resume_embedding.shape)

In [ ]:
# Compare Resume with Each Skill

semantic_skills = []

for skill, embedding in zip(skills, skill_embeddings):

    similarity = cosine_similarity(
        [embedding],
        resume_embedding
    )[0][0]

    if similarity > 0.30:
        semantic_skills.append((skill, similarity))

In [ ]:
# Sort the Results

semantic_skills = sorted(
    semantic_skills,
    key=lambda x: x[1],
    reverse=True
)

for skill, score in semantic_skills:
    print(f"{skill:30} {score:.3f}")

# Phase 5 : Intelligent Semantic Skill Extraction

Split Resume into Sentences

In [ ]:
sentences = []

doc = nlp(resume_text)

for sent in doc.sents:
    sentence = sent.text.strip()

    if len(sentence) > 5:
        sentences.append(sentence)

print("Total Sentences:", len(sentences))

In [ ]:
for i, sentence in enumerate(sentences):
    print(f"{i+1}. {sentence}")

Creating Embeddings for Every Sentence

In [ ]:
sentence_embeddings = model.encode(sentences)
print("Total Embeddings:", len(sentence_embeddings))

## Phase 5 Observation

Sentence Transformers were successfully integrated into the project.

An experiment was performed to semantically extract skills directly from the resume. However, due to the PDF extraction process merging multiple words (e.g., "usingpythonscikitlearntensorflow..."), semantic skill extraction produced incomplete results.

Instead of relying on semantic skill extraction, the project uses Sentence Transformers for Resume vs Job Description semantic similarity, which is more robust and closely matches how modern ATS systems evaluate resumes.

# Phase 6 : Resume vs Job Description Matching

In [ ]:
# Resume vs Job Description Matching

# Read Job Description

with open("sample_jd.txt", "r", encoding="utf-8") as file:
    job_description = file.read()

print(job_description)

Creating Job Description Embedding

In [ ]:
jd_embedding = model.encode([job_description])

print(jd_embedding.shape)

Creating Resume Embedding

In [ ]:
resume_embedding = model.encode([resume_text])

print(resume_embedding.shape)

Calculating Cosine Similarity

In [ ]:
similarity = cosine_similarity(
    resume_embedding,
    jd_embedding
)[0][0]

print("Similarity Score:", similarity)

Convert to Resume Match Score

In [ ]:
match_score = similarity * 100

print("=" * 50)
print(f"Resume Match Score : {match_score:.2f}%")
print("=" * 50)

# Phase 7 : ATS Score Calculation

Our ATS Formula

We'll use this weighted formula:

Component	Weight

Resume-JD Semantic Match	30

Skill Match	                40

Education	                10

Experience	                10

Resume Completeness	        10

Total	                    100


This is much closer to a real ATS than using semantic similarity alone.

In [ ]:
# Extract Skills from Job Description

jd_doc = nlp(job_description)

In [ ]:
matches = matcher(jd_doc)

jd_skills = []

for match_id, start, end in matches:
    skill = jd_doc[start:end].text.lower()

    if skill not in jd_skills:
        jd_skills.append(skill)

jd_skills = sorted(jd_skills)

print("Skills Found in Job Description:\n")

for skill in jd_skills:
    print(skill)

In [ ]:
# Compare Resume Skills with JD Skills

matched_skills = list(
    set(found_skills).intersection(set(jd_skills))
)

matched_skills = sorted(matched_skills)

print("Matched Skills:\n")

for skill in matched_skills:
    print(skill)

In [ ]:
# Finding Missing Skills
missing_skills = list(
    set(jd_skills) - set(found_skills)
)

missing_skills = sorted(missing_skills)

print("Missing Skills:\n")

for skill in missing_skills:
    print(skill)

In [ ]:
# Calculate Skill Match Score
skill_match = (len(matched_skills) / len(jd_skills)) * 100

print("=" * 50)
print(f"Skill Match Score : {skill_match:.2f}%")
print("=" * 50)

# Phase 8 : ATS Score Calculation

In [ ]:
# Semantic Score (30 Marks)

semantic_score = (match_score / 100) * 30
print(f"Semantic Score : {semantic_score:.2f}/30")

In [ ]:
# Calculating Skill Score (40 Marks)

skills_score = (skill_match / 100) * 40

print(f"Skill Score : {skills_score:.2f}/40")

In [ ]:
# Education Score 

education_score = 10

print(f"Education Score : {education_score}/10")

In [ ]:
# Experience Score
experience_score = 10

print(f"Experience Score : {experience_score}/10")

In [ ]:
# Resume Completeness 

completeness_score = 10

print(f"Resume Completeness : {completeness_score}/10")

In [ ]:
# Final ATS Score

ats_score = (
    semantic_score +
    skills_score +
    education_score +
    experience_score +
    completeness_score
)

print("=" * 60)
print(f"Final ATS Score : {ats_score:.2f}/100")
print("=" * 60)

# Phase 9 : Google Gemini AI Resume Suggestionsv

Instead of showing only:

ATS Score : 74.20%

our application will also generate an intelligent report such as:

Resume Analysis

ATS Score : 74.20%

Strengths:
 Strong Python skills
 Good NLP knowledge
 TensorFlow experience
 Git & GitHub

Missing Skills:
• Machine Learning
• Deep Learning
• Scikit-learn

Suggestions:
1. Add Machine Learning explicitly in the Skills section.
2. Mention Deep Learning projects.
3. Include Scikit-learn in Technical Skills.
4. Quantify achievements using numbers.
5. Add more AI Engineer keywords.

In [ ]:
# Install Google Gemini SDK

import sys
!{sys.executable} -m pip install -q google-generativeai

In [ ]:
# Importing the Library

# import google.generativeai as genai

print("Google Gemini Imported Successfully!")

In [ ]:
# Configure Gemini

from google import genai

client = genai.Client(api_key="YOUR_API_KEY")

print("Gemini Configured Successfully!")

In [ ]:
# Creating the Gemini Model 

MODEL_NAME = "gemini-2.5-flash"
print("Gemini Model Ready!")

In [ ]:
# Create the AI Prompt

prompt = f"""
You are an expert ATS Resume Reviewer.

Analyze the following resume against the job description.

Resume Match Score:
{match_score:.2f}%

Skill Match Score:
{skill_match:.2f}%

ATS Score:
{ats_score:.2f}/100

Matched Skills:
{matched_skills}

Missing Skills:
{missing_skills}

Resume:
{resume_text}

Job Description:
{job_description}

Please provide:

1. Overall Resume Review
2. Resume Strengths
3. Missing Skills
4. Weaknesses
5. Suggestions to Improve ATS Score
6. Final Recommendation
"""

In [ ]:
!pip install -U google-genai

In [ ]:
# Generating the AI Suggestions

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt
)
print(response.text)

# Phase 10 : Dynamic ATS Scoring

In [ ]:
education_keywords = [
    "bachelor",
    "b.sc",
    "btech",
    "b.tech",
    "computer science",
    "engineering",
    "master",
    "m.sc",
    "mtech"
]

education_score = 0

resume_lower = resume_text.lower()

for keyword in education_keywords:
    if keyword in resume_lower:
        education_score = 10
        break

print("=" * 50)
print(f"Education Score : {education_score}/10")
print("=" * 50)

In [ ]:
experience_keywords = [
    "intern",
    "internship",
    "experience",
    "worked",
    "employee",
    "project"
]

experience_score = 0

for keyword in experience_keywords:
    if keyword in resume_lower:
        experience_score = 10
        break

print("=" * 50)
print(f"Experience Score : {experience_score}/10")
print("=" * 50)

In [ ]:
# Checking whether the important resume sections are present

section_keywords = {
    "About": ["about", "aboutme"],
    "Technical Skills": ["technical skills", "technicalskills"],
    "Education": ["education"],
    "Projects": ["projects"],
    "Experience": ["experience", "workexperience", "intern"],
    "Certifications": ["certifications", "certification", "training"]
}

text = resume_text.lower()
count = 0

for keywords in section_keywords.values():
    for word in keywords:
        if word in text:
            count += 1
            break      # Count each section only once

score = (count / len(section_keywords)) * 10

print("-" * 50)
print("Sections Found:", count, "/", len(section_keywords))
print("Resume Completeness Score:", round(score, 2), "/10")
print("-" * 50)

In [ ]:
total_score = (
    semantic_score +
    skills_score +
    education_score +
    experience_score +
    completeness_score
)

print("-" * 60)
print("Final ATS Score :", round(total_score, 2), "/100")
print("-" * 60)

# Phase 11 – Build the Streamlit Web Application


In [ ]:
import os

os.chdir(r"C:\Users\user\Desktop\AI_Resume_Analyzer")

print(os.getcwd())

In [ ]:
from utils import *

In [ ]:
# Testing the complete pipeline

print(extract_text_from_pdf)
print(preprocess_text)
print(calculate_skill_match_score)
print(calculate_completeness_score)
print(calculate_ats_score)
print(generate_ai_suggestions)

In [ ]:
# test the actual resume 

resume_path = "sample_resume.pdf"

resume_text = extract_text_from_pdf(resume_path)

print("Resume text extracted successfully.")
print("Characters extracted:", len(resume_text))
print()
print(resume_text[:1000])

In [ ]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles in this folder:")
print(os.listdir())

In [ ]:
import os

pdf_path = os.path.abspath("sample_resume.pdf")

print(pdf_path)
print("Exists:", os.path.exists(pdf_path))
print("Is file:", os.path.isfile(pdf_path))
print("Size:", os.path.getsize(pdf_path), "bytes")

In [ ]:
resume_text = extract_text_from_pdf("sample_resume.pdf")

print("Resume text extracted successfully.")
print("Characters extracted:", len(resume_text))
print()
print(resume_text[:1500])

In [ ]:
clean_resume_text = preprocess_text(resume_text)

print("Preprocessing completed.")
print("Characters after preprocessing:", len(clean_resume_text))
print()
print(clean_resume_text[:1500])

In [ ]:
resume_skills = extract_skills(clean_resume_text)

print("Resume skills:")
print(resume_skills)

print("\nNumber of skills found:", len(resume_skills))

In [ ]:
with open("sample_jd.txt", "r", encoding="utf-8") as file:
    jd_text = file.read()

print("Job Description loaded successfully.")
print("Characters:", len(jd_text))
print()
print(jd_text[:1500])

In [ ]:
import os

jd_path = os.path.abspath("sample_jd.txt")

print(jd_path)
print("Exists:", os.path.exists(jd_path))
print("Is file:", os.path.isfile(jd_path))
print("Is directory:", os.path.isdir(jd_path))

if os.path.exists(jd_path) and os.path.isfile(jd_path):
    print("Size:", os.path.getsize(jd_path), "bytes")

In [ ]:
with open("sample_jd.txt", "r", encoding="utf-8") as file:
    jd_text = file.read()

print("Job Description loaded successfully.")
print("Characters:", len(jd_text))
print()
print(jd_text[:1500])

In [ ]:
clean_jd_text = preprocess_text(jd_text)

print("JD preprocessing completed.")
print("Characters after preprocessing:", len(clean_jd_text))
print()
print(clean_jd_text[:1500])

In [ ]:
jd_skills = extract_skills(clean_jd_text)

print("JD skills:")
print(jd_skills)

print("\nNumber of JD skills found:", len(jd_skills))

In [ ]:
matched_skills = semantic_skill_matching(
    clean_resume_text,
    jd_skills
)

print("Matched skills:")
print(matched_skills)

print("\nNumber of matched skills:", len(matched_skills))

In [ ]:
skill_match_score = calculate_skill_match_score(
    matched_skills,
    jd_skills
)

print("Skill Match Score:", skill_match_score)

In [ ]:
matched_skills = sorted(set(resume_skills) & set(jd_skills))
missing_skills = sorted(set(jd_skills) - set(resume_skills))

print("Matched skills:")
print(matched_skills)

print("\nMissing skills:")
print(missing_skills)

print("\nMatched:", len(matched_skills))
print("Missing:", len(missing_skills))

In [ ]:
for skill in ["machine learning", "deep learning",
              "natural language processing", "prompt engineering"]:
    print(skill, "->", skill in resume_text.lower())

In [ ]:
similarity_score = calculate_similarity(
    clean_resume_text,
    clean_jd_text
)

print("Semantic Similarity Score:", similarity_score)
print("Semantic Similarity Percentage:", similarity_score * 100)

In [ ]:
completeness_score, sections_found = calculate_completeness_score(
    clean_resume_text
)

print("Completeness Score:", completeness_score)
print("Sections Found:", sections_found)

In [ ]:
import py_compile

py_compile.compile("utils.py", doraise=True)

print("utils.py syntax is correct.")

In [ ]:
import importlib
import utils

importlib.reload(utils)

print("utils.py reloaded successfully.")

In [ ]:
ats_score = calculate_ats_score(
    similarity_score,
    skill_match_score,
    completeness_score
)

print("Final ATS Score:", round(ats_score, 2))

In [ ]:
print("similarity_score =", similarity_score)
print("skill_match_score =", skill_match_score)
print("completeness_score =", completeness_score)

In [ ]:
matched_skills = sorted(set(resume_skills) & set(jd_skills))

missing_skills = sorted(set(jd_skills) - set(resume_skills))

skill_match_score = calculate_skill_match_score(
    matched_skills,
    jd_skills
)

print("Matched skills:", matched_skills)
print("Missing skills:", missing_skills)
print("Skill Match Score:", skill_match_score)

In [ ]:
from utils import calculate_ats_score

In [ ]:
ats_score = calculate_ats_score(
    similarity_score,
    skill_match_score,
    completeness_score
)

print("Final ATS Score:", round(ats_score, 2))

In [ ]:
suggestions = generate_ai_suggestions(
    matched_skills,
    missing_skills,
    ats_score,
    similarity_score,
    completeness_score
)

print("AI Resume Suggestions:\n")

for i, suggestion in enumerate(suggestions, 1):
    print(f"{i}. {suggestion}")

In [ ]:
import os

print(os.getcwd())
print(os.listdir())

In [ ]:
import os

print(os.path.abspath("AI_Resume_Analyzer_.ipynb"))

In [98]:
import os
print(os.path.abspath("AI Resume Analyzer .ipynb"))

C:\Users\user\Desktop\AI_Resume_Analyzer\AI Resume Analyzer .ipynb
